# Stage 5-1. CLI Scripts

이 노트북은 `scripts/` 폴더의 CLI 진입점 4개를 Python에서 직접 호출하는 방법을 실습한다.

| 스크립트 | `main()` 역할 | 반환값 |
|---|---|---|
| `scripts/train.py` | `Experiment` 조립 → `exp.run()` → checkpoint 저장(선택) | epoch별 log 목록 |
| `scripts/evaluate.py` | `Experiment` 조립 → checkpoint 로드(선택) → `evaluator.evaluate()` | `{loss, metric, num_samples}` |
| `scripts/predict.py` | `Experiment` 조립 → checkpoint 로드(선택) → `predictor.predict(images)` | `{logits, predictions}` |
| `scripts/visualize.py` | `Experiment` 조립 → `exp.run()` → 학습 곡선·예측 grid 저장 | `{logs, log_path, pred_path}` |

각 스크립트는 `argparse`로 CLI 인수를 처리하지만, `main(args)` 시그니처를 통해 Python 객체로도 직접 호출할 수 있다.

**학습 목표**
1. `argparse.Namespace`를 직접 생성해 `main(args)`를 노트북에서 호출하는 패턴을 익힌다.
2. `train` → `evaluate` → `predict` → `visualize` 워크플로우를 순서대로 실행한다.
3. checkpoint를 통해 학습된 파라미터를 evaluate/predict 단계에서 재사용하는 흐름을 확인한다.
4. 3종 task에 대해 동일 패턴으로 스크립트를 호출한다.

## 0. 환경 설정

In [ ]:
import sys, os
sys.path.insert(0, "../..")

import argparse
import tempfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# CLI 스크립트 main 함수 임포트
from scripts.train     import main as train_main
from scripts.evaluate  import main as evaluate_main
from scripts.predict   import main as predict_main
from scripts.visualize import main as visualize_main

DATASET_DIR = "/mnt/d/datasets/mnist"
OUTPUT_DIR  = "../../outputs/stage5_demo"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. argparse.Namespace — 노트북에서 args 직접 생성

In [ ]:
# CLI에서는 python scripts/train.py --task multiclass --epochs 3
# 노트북에서는 Namespace 객체를 직접 생성한다

args = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    epochs      = 3,
    batch_size  = 64,
    lr          = 0.01,
    seed        = 42,
    dataset_dir = DATASET_DIR,
    checkpoint  = None,
)

print("args 타입  :", type(args))
print("args.task  :", args.task)
print("args.epochs:", args.epochs)
print()
print("CLI 동등 명령:")
print(f"  python scripts/train.py --task {args.task} --model {args.model} --epochs {args.epochs} --lr {args.lr}")

## 2. train.main() — 학습 실행 + checkpoint 저장

In [ ]:
ck_path = os.path.join(OUTPUT_DIR, "multiclass_mlp")

train_args = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    epochs      = 3,
    batch_size  = 64,
    lr          = 0.01,
    seed        = 42,
    dataset_dir = DATASET_DIR,
    checkpoint  = ck_path,
)

print("=== train.main() 실행 ===")
train_logs = train_main(train_args)

print(f"\n반환값 타입    : {type(train_logs)}")
print(f"log 개수       : {len(train_logs)} (epochs)")
print(f"첫 번째 log 키 : {list(train_logs[0].keys())}")
print(f"checkpoint 저장: {ck_path}.npz")

## 3. evaluate.main() — checkpoint 로드 후 평가

In [ ]:
eval_args = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    batch_size  = 64,
    seed        = 42,
    dataset_dir = DATASET_DIR,
    checkpoint  = ck_path,
)

print("=== evaluate.main() 실행 ===")
eval_result = evaluate_main(eval_args)

print(f"\n반환값: {eval_result}")

> **checkpoint 미사용 vs 사용 비교**: checkpoint 없이 `evaluate`를 호출하면 초기화된 가중치로 평가하므로 결과가 낮다.

In [ ]:
# checkpoint 없이 평가 (무작위 초기화)
eval_args_no_ck = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    batch_size  = 64,
    seed        = 42,
    dataset_dir = DATASET_DIR,
    checkpoint  = None,
)

print("checkpoint 없이 (초기화 상태):")
result_no_ck = evaluate_main(eval_args_no_ck)
print()
print("checkpoint 사용 (학습된 파라미터):")
result_with_ck = evaluate_main(eval_args)

print(f"\n  metric 향상: {result_no_ck['metric']:.4f} → {result_with_ck['metric']:.4f}")

## 4. predict.main() — 개별 샘플 예측

In [ ]:
predict_args = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    seed        = 42,
    dataset_dir = DATASET_DIR,
    checkpoint  = ck_path,
    n           = 8,
)

print("=== predict.main() 실행 ===")
pred_result = predict_main(predict_args)

print(f"\n반환값 키           : {list(pred_result.keys())}")
print(f"predictions shape  : {pred_result['predictions'].shape}")
print(f"logits shape       : {pred_result['logits'].shape}")

## 5. visualize.main() — 학습 곡선 + 예측 grid 저장

In [ ]:
viz_args = argparse.Namespace(
    task        = "multiclass",
    model       = "mlp",
    epochs      = 3,
    batch_size  = 64,
    lr          = 0.01,
    seed        = 42,
    dataset_dir = DATASET_DIR,
    output_dir  = OUTPUT_DIR,
    n_samples   = 16,
)

print("=== visualize.main() 실행 ===")
viz_result = visualize_main(viz_args)

print(f"\n반환값 키    : {list(viz_result.keys())}")
print(f"log_path    : {viz_result['log_path']}")
print(f"pred_path   : {viz_result['pred_path']}")

In [ ]:
# 저장된 이미지를 노트북에서 표시
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("visualize.main() 출력 결과", fontsize=13, fontweight="bold")

for ax, key, title in [
    (axes[0], "log_path",  "Training Log"),
    (axes[1], "pred_path", "Predictions"),
]:
    img = mpimg.imread(viz_result[key])
    ax.imshow(img)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. train → evaluate → predict 전체 워크플로우

In [ ]:
def run_workflow(task, epochs=3, lr=0.01, n_predict=8):
    """train → evaluate → predict 3단계 워크플로우를 실행한다."""
    ck = os.path.join(OUTPUT_DIR, f"{task}_mlp")

    # 1. Train
    logs = train_main(argparse.Namespace(
        task=task, model="mlp", epochs=epochs, batch_size=64,
        lr=lr, seed=42, dataset_dir=DATASET_DIR, checkpoint=ck,
    ))
    last = logs[-1]
    print(f"[{task}] train → test  loss={last['test']['loss']:.4f}  metric={last['test']['metric']:.4f}")

    # 2. Evaluate
    ev = evaluate_main(argparse.Namespace(
        task=task, model="mlp", batch_size=64,
        seed=42, dataset_dir=DATASET_DIR, checkpoint=ck,
    ))

    # 3. Predict
    pr = predict_main(argparse.Namespace(
        task=task, model="mlp", seed=42,
        dataset_dir=DATASET_DIR, checkpoint=ck, n=n_predict,
    ))

    return logs, ev, pr

print("="*60)

## 7. 3종 task 워크플로우 실행

In [ ]:
results = {}
for task in ["multiclass", "binary", "regression"]:
    print(f"\n{'='*50}")
    print(f"Task: {task}")
    print(f"{'='*50}")
    logs, ev, pr = run_workflow(task, epochs=3)
    results[task] = {"logs": logs, "eval": ev, "pred": pr}

In [ ]:
# 최종 결과 요약
metric_labels = {"multiclass": "Accuracy", "binary": "Binary Acc", "regression": "R²"}

print(f"\n{'='*60}")
print(f"{'task':<14} | {'train loss':>10} {'train metric':>13} | {'test loss':>10} {'test metric':>12}")
print("-"*60)
for task in ["multiclass", "binary", "regression"]:
    last = results[task]["logs"][-1]
    label = metric_labels[task]
    print(f"{task:<14} | {last['train']['loss']:>10.4f} {last['train']['metric']:>13.4f} | "
          f"{last['test']['loss']:>10.4f} {last['test']['metric']:>12.4f}")

## 8. 학습 곡선 비교 시각화

In [ ]:
colors = {"multiclass": "steelblue", "binary": "#E87B4C", "regression": "mediumseagreen"}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
fig.suptitle("3 Tasks — CLI Scripts Workflow", fontsize=13, fontweight="bold")

for col, task in enumerate(["multiclass", "binary", "regression"]):
    logs = results[task]["logs"]
    epochs    = [l["epoch"]           for l in logs]
    tr_loss   = [l["train"]["loss"]   for l in logs]
    te_loss   = [l["test"]["loss"]    for l in logs]
    tr_metric = [l["train"]["metric"] for l in logs]
    te_metric = [l["test"]["metric"]  for l in logs]
    c = colors[task]

    axes[0, col].plot(epochs, tr_loss, label="train", color=c,    linewidth=2, marker="o")
    axes[0, col].plot(epochs, te_loss, label="test",  color=c,    linewidth=2, marker="o", linestyle="--", alpha=0.6)
    axes[0, col].set_title(f"{task} — Loss")
    axes[0, col].set_xlabel("epoch")
    axes[0, col].set_ylabel("loss")
    axes[0, col].legend(fontsize=8)
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(epochs, tr_metric, label="train", color=c,  linewidth=2, marker="o")
    axes[1, col].plot(epochs, te_metric, label="test",  color=c,  linewidth=2, marker="o", linestyle="--", alpha=0.6)
    axes[1, col].set_title(f"{task} — {metric_labels[task]}")
    axes[1, col].set_xlabel("epoch")
    axes[1, col].set_ylabel(metric_labels[task])
    axes[1, col].legend(fontsize=8)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 스크립트 CLI 실행 명령 참조

In [ ]:
print("=== CLI 실행 명령 참조 ===")
print()
print("# 학습 (multiclass MLP, 10 epochs, checkpoint 저장)")
print("conda run -n numpy_py311 python scripts/train.py \\")
print("    --task multiclass --model mlp --epochs 10 --lr 0.01 \\")
print("    --checkpoint outputs/multiclass/mlp/model")
print()
print("# 평가 (checkpoint 로드)")
print("conda run -n numpy_py311 python scripts/evaluate.py \\")
print("    --task multiclass --model mlp \\")
print("    --checkpoint outputs/multiclass/mlp/model")
print()
print("# 예측 (16개 샘플)")
print("conda run -n numpy_py311 python scripts/predict.py \\")
print("    --task multiclass --model mlp --n 16 \\")
print("    --checkpoint outputs/multiclass/mlp/model")
print()
print("# 시각화 (학습 + 이미지 저장)")
print("conda run -n numpy_py311 python scripts/visualize.py \\")
print("    --task multiclass --model mlp --epochs 10 \\")
print("    --output_dir outputs/multiclass/mlp")

## 10. 정리

**스크립트별 역할 분담**

```
scripts/train.py
  args → build_config() → Experiment(config) → exp.run() → checkpoints.save()
  반환: epoch별 log 목록

scripts/evaluate.py
  args → build_config() → Experiment(config) → checkpoints.load() → evaluator.evaluate()
  반환: {loss, metric, num_samples}

scripts/predict.py
  args → build_config() → Experiment(config) → checkpoints.load() → predictor.predict(images)
  반환: {logits, predictions}

scripts/visualize.py
  args → build_config() → Experiment(config) → exp.run()
       → plot_training_log() → Visualizer.plot_predictions()
  반환: {logs, log_path, pred_path}
```

**노트북 호출 패턴**
```python
from scripts.train import main as train_main
import argparse

args = argparse.Namespace(task="multiclass", model="mlp", epochs=5, ...)
logs = train_main(args)
```

**핵심 설계 원칙**
- 각 스크립트는 `parse_args()` / `build_config()` / `main()` 3단계로 분리되어 CLI와 Python 호출 모두를 지원한다.
- `Experiment`가 모든 조립을 담당하므로 스크립트는 config 전달과 결과 출력에만 집중한다.
- checkpoint 없이 evaluate/predict를 실행하면 초기화된 파라미터로 동작하므로 의미 있는 비교를 위해 checkpoint를 명시해야 한다.